In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

# -----------------------------
# Experiment metadata
# -----------------------------
experiment_name = "exp12_pls_component_search_20260323"

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# -----------------------------
# Features
# -----------------------------
spectral_cols = [
    c for c in train.columns
    if c not in ["sample number","species number","樹種","含水率"]
]

X = train[spectral_cols].values
y = train["含水率"].values
X_test = test[spectral_cols].values

print("Spectral features:", X.shape[1])

# -----------------------------
# Scaling (critical for PLS)
# -----------------------------
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# -----------------------------
# Cross validation search
# -----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

print("\nSearching PLS components...\n")

for n in range(2, 41):

    rmse_scores = []

    for train_idx, val_idx in kf.split(X_scaled):

        X_train = X_scaled[train_idx]
        X_val = X_scaled[val_idx]

        y_train = y[train_idx]
        y_val = y[val_idx]

        model = PLSRegression(n_components=n)

        model.fit(X_train, y_train)

        preds = model.predict(X_val).flatten()

        rmse = np.sqrt(mean_squared_error(y_val, preds))

        rmse_scores.append(rmse)

    mean_rmse = np.mean(rmse_scores)

    results.append((n, mean_rmse))

    print(f"Components={n:<3} RMSE={mean_rmse:.4f}")

# -----------------------------
# Best components
# -----------------------------
best_n = min(results, key=lambda x: x[1])[0]

print("\nBest n_components:", best_n)

# -----------------------------
# Train final model
# -----------------------------
model = PLSRegression(n_components=best_n)

model.fit(X_scaled, y)

test_preds = model.predict(X_test_scaled).flatten()

print("Sample predictions:", test_preds[:10])

# -----------------------------
# Save submission
# -----------------------------
os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

output_path = f"../submissions/{experiment_name}.csv"

submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved:", output_path)

check = pd.read_csv(output_path, header=None)
print(check.head())

Train shape: (1322, 1559)
Test shape: (550, 1558)
Spectral features: 1555

Searching PLS components...

Components=2   RMSE=30.7387
Components=3   RMSE=26.7481
Components=4   RMSE=23.2880
Components=5   RMSE=22.6631
Components=6   RMSE=21.0534
Components=7   RMSE=19.7771
Components=8   RMSE=19.1445
Components=9   RMSE=18.5922
Components=10  RMSE=17.9392
Components=11  RMSE=17.3292
Components=12  RMSE=16.9735
Components=13  RMSE=16.5819
Components=14  RMSE=15.9839
Components=15  RMSE=15.3163
Components=16  RMSE=15.1054
Components=17  RMSE=14.7324
Components=18  RMSE=14.2503
Components=19  RMSE=13.1080
Components=20  RMSE=13.0988
Components=21  RMSE=12.8996
Components=22  RMSE=12.4078
Components=23  RMSE=12.1618
Components=24  RMSE=12.4328
Components=25  RMSE=12.4006
Components=26  RMSE=13.0343
Components=27  RMSE=12.9770
Components=28  RMSE=13.8316
Components=29  RMSE=14.3567
Components=30  RMSE=14.3082
Components=31  RMSE=14.2786
Components=32  RMSE=14.1476
Components=33  RMSE=13.2113
